**Hotel-Review-System**

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("tripadvisor_hotel_reviews.csv")
df

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5
...,...,...
20486,"best kept secret 3rd time staying charm, not 5...",5
20487,great location price view hotel great quick pl...,4
20488,"ok just looks nice modern outside, desk staff ...",2
20489,hotel theft ruined vacation hotel opened sept ...,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20491 entries, 0 to 20490
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  20491 non-null  object
 1   Rating  20491 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 320.3+ KB


In [6]:
df.describe()

,Rating
count,20491.000000
mean,3.952223
std,1.233030
min,1.000000
25%,3.000000
50%,4.000000
75%,5.000000
max,5.000000


In [7]:
df['Rating'].value_counts()

Rating
5    9054
4    6039
3    2184
2    1793
1    1421
Name: count, dtype: int64

**Cleaning Data**

In [8]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [33]:
import re 
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from string import punctuation
import unicodedata

def clean(text):
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')
    text = re.sub(r'-?\b\d+(\.\d+)?\b|-?\b\.\d+\b', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b[^a-zA-Z\s]+\b', ' ', text)  
    text = re.sub(r'\b(\w)\1{2,}\w*\b', ' ', text)
    text = re.sub(r'\b(?=\w*\d)(?=\w*[a-zA-Z])\w+\b', '', text)
    text = text.lower()
    punc = list(punctuation)
    tokens = word_tokenize(text)
    stop = list(stopwords.words("english"))+punc
    words = [words for words in tokens if words not in stop]
    lemm = WordNetLemmatizer()
    cleaned = [lemm.lemmatize(word) for word in words]
    return ' '.join(cleaned)


In [34]:
df['clean_rev'] = df['Review'].apply(clean)

In [11]:
df.head()

,Review,Rating,clean_rev
0,nice hotel expensive parking got good deal sta...,4,nice hotel expensive parking got good deal sta...
1,ok nothing special charge diamond member hilto...,2,ok nothing special charge diamond member hilto...
2,nice rooms not 4* experience hotel monaco seat...,3,nice room experience hotel monaco seattle good...
3,"unique, great stay, wonderful time hotel monac...",5,unique great stay wonderful time hotel monaco ...
4,"great stay great stay, went seahawk game aweso...",5,great stay great stay went seahawk game awesom...


In [12]:
df.isna().sum()

Review       0
Rating       0
clean_rev    0
dtype: int64

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfid_ng = TfidfVectorizer(
    ngram_range=(1,3),
    lowercase=True,
)
X = tfid_ng.fit_transform(df['clean_rev'])
voc = tfid_ng.get_feature_names_out(df['clean_rev'])

In [14]:
voc

array(['__c', '__c __c', '__c __c hurry', ..., 'zz', 'zz near',
       'zz near la'], shape=(2630813,), dtype=object)

In [36]:
y = df['Rating']

In [44]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,train_size=0.2)

In [ ]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.metrics import r2_score,mean_absolute_error,mean_absolute_percentage_error,mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

results = []


models = {
    "Linear Regression" :LinearRegression(),
    'Ridge':Ridge(),
    'LassoRegression':Lasso(),
    'KNeighborsRegressor' : KNeighborsRegressor()
    
}
for model in models.values():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    results.append({
        'Model': model.__class__.__name__,
        'MSE': mse,
        'R2': r2,
        'MAE': mae,
        'MAPE': mape
    })

metrics_df = pd.DataFrame(results)
print(metrics_df)

                 Model       MSE        R2       MAE      MAPE
0     LinearRegression  0.702964  0.539054  0.670169  0.277391
1                Ridge  0.852976  0.440689  0.738557  0.314767
2                Lasso  1.525865 -0.000537  0.943630  0.421805
3  KNeighborsRegressor  1.057595  0.306516  0.774904  0.330100


In [51]:
models['LinearRegression'] = LinearRegression().fit(X_train, y_train)

In [66]:
import pickle
lr_smote = pickle.load(open('lr.pkl','rb'))

c:\Users\Adnan\Desktop\ML_Models\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.4.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [75]:
# Get user input
text = input("Enter review: ")

# Clean the review text
clean_data = clean(text)  # This should return a cleaned string

# Vectorize using the already trained vectorizer (do NOT use fit_transform)
X = tfid_ng.transform([clean_data])  # Wrap in a list to avoid error

# Predict using the trained model
predicted_rating_lr = models['LinearRegression'].predict(X)
predicted_rating_smote = lr_smote.predict(X)

# Output the result (using LinearRegression prediction)
clamped_rating_lr = max(0, min(5, predicted_rating_lr[0]))
clamped_rating_lrsm = max(0, min(5, predicted_rating_smote[0]))

print(f"The predicted rating for the review by lr :'{text}' = {clamped_rating_lr:.2f}")
print(f"The predicted rating for the review by smote :'{text}' = {clamped_rating_lrsm:.2f}")


The predicted rating for the review by lr :'	“Decent for the price. Could use some renovations, but it’s okay.”' = 3.17
The predicted rating for the review by smote :'	“Decent for the price. Could use some renovations, but it’s okay.”' = 2.44
